# Shipment consolidation

Minimize truck linehaul costs + extra-stop charges + LTL costs while serving every shipment and respecting all operating rules.

**Approach:** Generate feasible truck routes, then use Gurobi to select a combination of routes and LTL shipments.

**Process:** Load data → calculate the all-LTL baseline → generate and check routes → formulate and solve route selection → independently validate the plan → report savings.


## 1. Setup

Run from the folder containing this notebook and the Excel files. Run cells from top to bottom.


In [1]:
# Run only if packages are missing:
# %pip install pandas openpyxl gurobipy

from pathlib import Path

import pandas as pd
import gurobipy as gp
from gurobipy import GRB


## 2. Load the simplified data

Start with 60 shipments over 3 days. Each Excel sheet holds one input table.


In [2]:
DATA_FILE = Path("project1_simplified.xlsx")
data = pd.read_excel(DATA_FILE, sheet_name=None, engine="openpyxl")
shipments = data["shipments"]

print(f"{len(shipments)} shipments across {shipments['day'].nunique()} days")
shipments.head()


60 shipments across 3 days


,shipment_id,origin_id,destination_id,day,pickup_open_hour,pickup_close_hour,delivery_open_hour,delivery_close_hour,weight_lb,linear_feet,pallets,freight_class,temperature,compatibility_group,pickup_service_hours,delivery_service_hours
0,S001,F01,F03,1,8.0,10.0,12.0,17.0,22000,16,8,70,ambient,general,0.5,0.5
1,S002,F02,F04,1,9.0,12.0,12.0,16.0,9000,24,12,125,ambient,general,0.5,0.5
2,S003,F08,F05,1,8.0,11.0,12.0,17.0,18000,14,8,70,ambient,general,0.5,0.5
3,S004,F11,F09,1,8.5,11.5,12.5,17.0,8000,22,10,125,ambient,general,0.5,0.5
4,S005,F06,F07,1,7.5,11.0,12.0,17.0,16000,15,7,85,ambient,general,0.5,0.5


## 3. All-LTL baseline

Calculate the cost of sending every shipment through LTL, including fuel surcharges. This is the total cost our optimized plan will try to beat.


In [3]:
distances = data["distance_matrix"].set_index(
    ["from_id", "to_id"]
)["distance_miles"]
if not distances.index.is_unique:
    raise ValueError("Duplicate origin-destination pairs in distance_matrix.")

rates = data["ltl_rates"]
ltl_cost = {}

for shipment in shipments.itertuples(index=False):
    miles = distances.at[(shipment.origin_id, shipment.destination_id)]
    matching_rates = rates[
        (rates["freight_class"] == shipment.freight_class)
        & (rates["weight_from_lb"] <= shipment.weight_lb)
        & (shipment.weight_lb < rates["weight_to_lb"])
        & (rates["distance_from_miles"] <= miles)
        & (miles < rates["distance_to_miles"])
    ]
    if len(matching_rates) != 1:
        raise ValueError(f"Expected one LTL rate for {shipment.shipment_id}; found {len(matching_rates)}.")

    rate = matching_rates.iloc[0]
    ltl_cost[shipment.shipment_id] = rate["base_cost_cad"] * (1 + rate["fuel_surcharge_pct"])

all_ltl_baseline = sum(ltl_cost.values())
print(f"All-LTL baseline: CAD ${all_ltl_baseline:,.2f}")


All-LTL baseline: CAD $93,298.35


## 4. Prepare route generation

Use dictionaries for quick lookups and read the operating limits from the workbook. The same code handles the simplified or main file; change `DATA_FILE` above and rerun all cells to switch.

In [4]:
from collections import Counter
from time import perf_counter
from IPython.display import display
import json

started = perf_counter()
S = data["shipments"].set_index("shipment_id").to_dict("index")
F = data["facilities"].set_index("facility_id").to_dict("index")
K = data["tractors"].set_index("tractor_id").to_dict("index")
E = data["equipment"].set_index("equipment_id").to_dict("index")
D = data["depots"].set_index("depot_id").to_dict("index")
C = data["tl_costs"].set_index("equipment_id").to_dict("index")
rules = data["driver_rules"].set_index("parameter")["value"].to_dict()
settings = data["parameters"].set_index("parameter")["value"].to_dict()
travel = {(r.from_id, r.to_id): (r.distance_miles, r.drive_hours)
          for r in data["distance_matrix"].itertuples(index=False)}
for location in list(F) + list(D):
    travel[location, location] = (0.0, 0.0)

dimensions = [("weight_lb", "weight_capacity_lb"),
              ("linear_feet", "length_capacity_feet"), ("pallets", "pallet_capacity")]
days = sorted({int(s["day"]) for s in S.values()})
max_shipments = int(settings["max_stops"]) // 2
need_breaks = rules["max_drive_hours"] > rules["break_after_drive_hours"]

# The supplied repeating shifts guarantee the required rest between days.
for k, truck in K.items():
    if 24 + truck["shift_start_hour"] - truck["shift_end_hour"] < rules["min_rest_hours"]:
        raise ValueError(f"{k}: changed shifts require additional between-day rest constraints.")
for sheet, key in [("shipments", "shipment_id"), ("tractors", "tractor_id"),
                   ("facilities", "facility_id"), ("depots", "depot_id"),
                   ("equipment", "equipment_id"), ("tl_costs", "equipment_id")]:
    if not data[sheet][key].is_unique:
        raise ValueError(f"Duplicate identifiers in {sheet}.")
if any((a, b) not in travel for a in list(F) + list(D) for b in list(F) + list(D)):
    raise ValueError("Missing directed travel lanes.")

# Each event is (shipment ID, P for pickup or D for delivery).
events = {}
for i, shipment in S.items():
    for kind, prefix, field in [("P", "pickup", "origin_id"), ("D", "delivery", "destination_id")]:
        location = shipment[field]
        events[i, kind] = {
            "location": location,
            "open": max(shipment[prefix + "_open_hour"], F[location]["open_hour"]),
            "close": min(shipment[prefix + "_close_hour"], F[location]["close_hour"]),
            "service": shipment[prefix + "_service_hours"],
        }

### Schedule and price one route

Follow its stops, tracking time and the freight on board; reject any rule violation. Breaks are recorded separately from handling, and compatibility is checked only for freight carried together.

In [5]:
def make_route(sequence, tractor, day):
    truck = K[tractor]
    equipment = E[truck["equipment_id"]]
    home = truck["home_depot_id"]
    ids = tuple(sorted(i for i, kind in sequence if kind == "P"))
    if not ids or len(set(ids)) != len(ids) or len(sequence) > settings["max_stops"]:
        return None
    if any(S[i]["day"] != day for i, kind in sequence):
        return None

    first = events[sequence[0]]
    departure = max(truck["shift_start_hour"], D[home]["open_hour"],
                    first["open"] - travel[home, first["location"]][1])
    now, location = departure, home
    miles = driving = since_break = 0.0
    onboard, delivered, actions = [], set(), []
    finish_by = min(truck["shift_end_hour"], D[home]["close_hour"], 24)

    # Include the final empty journey home in the same loop.
    for event in list(sequence) + [None]:
        target = home if event is None else events[event]["location"]
        distance, hours = travel[location, target]
        if hours > rules["break_after_drive_hours"] or driving + hours > rules["max_drive_hours"]:
            return None
        if since_break + hours > rules["break_after_drive_hours"]:
            end = now + rules["break_hours"]
            actions.append(dict(kind="break", location=location, start=now, end=end))
            now, since_break = end, 0.0
        actions.append(dict(kind="drive", origin=location, destination=target,
                            start=now, end=now + hours))
        now += hours
        miles += distance
        driving += hours
        since_break += hours
        location = target
        if now > finish_by:
            return None
        if event is None:
            break

        i, kind = event
        stop = events[event]
        start = max(now, stop["open"])
        end = start + stop["service"]
        if end > min(stop["close"], finish_by):
            return None
        # A sufficiently long wait can contain explicitly recorded off-duty rest.
        if need_breaks and since_break > 0 and start - now >= rules["break_hours"]:
            rest_end = now + rules["break_hours"]
            actions.append(dict(kind="break", location=location, start=now, end=rest_end))
            now, since_break = rest_end, 0.0
        if start > now:
            actions.append(dict(kind="wait", location=location, start=now, end=start))
        if kind == "P":
            if i in onboard or i in delivered:
                return None
            if S[i]["temperature"] == "chilled" and not equipment["chilled_capable"]:
                return None
            if any(S[j]["temperature"] != S[i]["temperature"] or
                   S[j]["compatibility_group"] != S[i]["compatibility_group"] for j in onboard):
                return None
            onboard.append(i)
        else:
            if i not in onboard or (settings["lifo_required"] and onboard[-1] != i):
                return None
            onboard.remove(i)
            delivered.add(i)
        if any(sum(S[j][field] for j in onboard) > equipment[cap] for field, cap in dimensions):
            return None
        actions.append(dict(kind=kind, shipment=i, location=location, start=start, end=end))
        now = end

    if onboard or delivered != set(ids) or now - departure > rules["max_duty_hours"]:
        return None
    if 24 * day + now >= 24 * settings["horizon_end_day"]:
        return None
    price = C[truck["equipment_id"]]
    linehaul = max(price["minimum_linehaul_cad"], miles * price["linehaul_cad_per_mile"])
    stop_cost = max(0, len(sequence) - 2) * price["extra_stop_cad"]
    return dict(tractor=tractor, day=int(day), shipments=ids, sequence=tuple(sequence),
                departure=departure, return_time=now, miles=miles, drive_hours=driving,
                duty_hours=now - departure, linehaul=linehaul, stop_cost=stop_cost,
                cost=linehaul + stop_cost, actions=actions)

## 5. Generate candidate routes

Start with single shipments, then insert another pickup and delivery in different positions. To limit runtime, explore nearby partners and expand a limited number of promising routes; this is a heuristic, so some feasible routes may be missed.

In [6]:
PARTNER_LIMIT = 12  # Nearby shipments to try per route; increase for a broader search.
EXPAND_LIMIT = 60   # Routes to expand at each size (all single-shipment routes are expanded).

def saving(route):
    return sum(ltl_cost[i] for i in route["shipments"]) - route["cost"]

def proximity(i, j):
    return (travel[S[i]["origin_id"], S[j]["origin_id"]][0]
            + travel[S[i]["destination_id"], S[j]["destination_id"]][0])

# Trucks with identical equipment, depot and shifts can reuse the same candidates.
profiles = {}
for k, truck in K.items():
    key = tuple(truck[field] for field in ["home_depot_id", "equipment_id",
                                          "shift_start_hour", "shift_end_hour"])
    profiles.setdefault(key, []).append(k)

routes = []
generation_start = perf_counter()
for day in days:
    today = [i for i in S if S[i]["day"] == day]
    neighbours = {i: sorted((j for j in today if j != i), key=lambda j: (proximity(i, j), j))
                 [:PARTNER_LIMIT] for i in today}
    for trucks in profiles.values():
        tractor = trucks[0]
        frontier = [make_route(((i, "P"), (i, "D")), tractor, day) for i in today]
        frontier = [route for route in frontier if route is not None]
        best = {route["shipments"]: route for route in frontier}
        for size in range(2, max_shipments + 1):
            expanded, tried = [], set()
            for route in frontier:
                partners = set(j for i in route["shipments"] for j in neighbours[i]) - set(route["shipments"])
                partners = sorted(partners, key=lambda j: (min(proximity(i, j) for i in route["shipments"]), j))[:PARTNER_LIMIT]
                sequence = route["sequence"]
                for j in partners:
                    for pickup in range(len(sequence) + 1):
                        with_pickup = sequence[:pickup] + ((j, "P"),) + sequence[pickup:]
                        for delivery in range(pickup + 1, len(with_pickup) + 1):
                            order = with_pickup[:delivery] + ((j, "D"),) + with_pickup[delivery:]
                            if order in tried:
                                continue
                            tried.add(order)
                            candidate = make_route(order, tractor, day)
                            if candidate is None:
                                continue
                            expanded.append(candidate)
                            group = candidate["shipments"]
                            if group not in best or candidate["cost"] < best[group]["cost"]:
                                best[group] = candidate
            # Keep costly intermediate routes eligible: adding freight can make them useful.
            frontier = sorted(expanded, key=lambda r: (-saving(r), r["sequence"]))[:EXPAND_LIMIT]
            if not frontier:
                break
        for route in best.values():
            # Completed routes costing more than their all-LTL alternative cannot help.
            if saving(route) > 0:
                for k in trucks:
                    routes.append(dict(route, tractor=k))
    print(f"Day {day}: candidates accumulated = {len(routes):,}")

generation_seconds = perf_counter() - generation_start
print(f"Generated {len(routes):,} candidates in {generation_seconds:.1f} seconds.")
# An empty candidate pool is valid: Gurobi can send every shipment through LTL.

Day 1: candidates accumulated = 508


Day 2: candidates accumulated = 1,036


Day 3: candidates accumulated = 1,508
Generated 1,508 candidates in 0.3 seconds.


## 6. Select routes with Gurobi

Use the class pattern: binary decision variables, a linear cost objective, and assignment constraints. Each candidate already has a feasible schedule and a known cost, including stop-off charges.

- $x_r=1$ if route $r$ is selected; $y_i=1$ if shipment $i$ uses LTL.
- Minimize $\sum_r c_r x_r + \sum_i l_i y_i$.
- For every shipment: routes carrying it + its LTL choice = 1.
- For every truck and day: selected routes using that truck ≤ 1.

In [7]:
R, I = range(len(routes)), list(S)
shipment_routes = {i: [] for i in I}
truck_routes = {(k, day): [] for k in K for day in days}
for r, route in enumerate(routes):
    for i in route["shipments"]:
        shipment_routes[i].append(r)
    truck_routes[route["tractor"], route["day"]].append(r)

model = gp.Model("Shipment_consolidation")
x = model.addVars(R, vtype=GRB.BINARY, name="select_route")
y = model.addVars(I, vtype=GRB.BINARY, name="use_ltl")
model.setObjective(gp.quicksum(routes[r]["cost"] * x[r] for r in R)
                   + gp.quicksum(ltl_cost[i] * y[i] for i in I), GRB.MINIMIZE)
for i in I:
    model.addConstr(gp.quicksum(x[r] for r in shipment_routes[i]) + y[i] == 1)
for k, day in truck_routes:
    model.addConstr(gp.quicksum(x[r] for r in truck_routes[k, day]) <= 1)

# Always provide a feasible starting solution: all shipments go through LTL.
for r in R:
    x[r].Start = 0
for i in I:
    y[i].Start = 1
model.Params.TimeLimit = 120
model.Params.MIPGap = 0
model.Params.OutputFlag = 0
model.optimize()
if model.SolCount == 0:
    raise RuntimeError(f"No solution available; Gurobi status {model.Status}.")

selected_routes = [routes[r] for r in R if x[r].X > 0.5]
ltl_shipments = [i for i in I if y[i].X > 0.5]
print(f"Selected {len(selected_routes)} truck routes; {len(ltl_shipments)} shipments use LTL.")
print(f"Solver gap within candidate pool: {model.MIPGap:.2%}")

Set parameter Username


Set parameter LicenseID to value 2859653


Academic license - for non-commercial use only - expires 2027-09-02


Set parameter TimeLimit to value 120


Set parameter MIPGap to value 0


Selected 24 truck routes; 8 shipments use LTL.
Solver gap within candidate pool: 0.00%


## 7. Independently check the plan

Read the selected schedule against the original workbook tables, without calling the route generator. Recalculate costs and verify service, timing, loading, driver rules and truck availability before reporting savings.

In [8]:
def check_plan(plan, external, tables):
    def require(condition, message):
        if not condition:
            raise ValueError("Plan check failed: " + message)

    # Fresh lookups: do not trust the generator's cached costs, loads or totals.
    shipments_check = tables["shipments"].set_index("shipment_id").to_dict("index")
    trucks_check = tables["tractors"].set_index("tractor_id").to_dict("index")
    equipment_check = tables["equipment"].set_index("equipment_id").to_dict("index")
    facilities_check = tables["facilities"].set_index("facility_id").to_dict("index")
    depots_check = tables["depots"].set_index("depot_id").to_dict("index")
    prices = tables["tl_costs"].set_index("equipment_id").to_dict("index")
    limits = tables["driver_rules"].set_index("parameter")["value"].to_dict()
    parameters = tables["parameters"].set_index("parameter")["value"].to_dict()
    lanes = {(r.from_id, r.to_id): (r.distance_miles, r.drive_hours)
             for r in tables["distance_matrix"].itertuples(index=False)}
    for location in list(facilities_check) + list(depots_check):
        lanes[location, location] = (0, 0)
    external_costs = {}
    for i, s in shipments_check.items():
        distance = lanes[s["origin_id"], s["destination_id"]][0]
        matches = [r for r in tables["ltl_rates"].itertuples(index=False)
                   if r.freight_class == s["freight_class"]
                   and r.weight_from_lb <= s["weight_lb"] < r.weight_to_lb
                   and r.distance_from_miles <= distance < r.distance_to_miles]
        require(len(matches) == 1, f"LTL rate for {i}")
        external_costs[i] = matches[0].base_cost_cad * (1 + matches[0].fuel_surcharge_pct)

    served, used, tours = Counter(external), set(), []
    previous_tours = {}
    weighted_weight = weighted_feet = total_miles = 0.0
    for route in plan:
        k, day = route["tractor"], route["day"]
        require(k in trucks_check and 1 <= day <= parameters["demand_days"], "truck/day")
        require((k, day) not in used, f"truck {k} used twice on day {day}")
        used.add((k, day))
        truck = trucks_check[k]
        equipment = equipment_check[truck["equipment_id"]]
        home = truck["home_depot_id"]
        depot = depots_check[home]
        now, location = route["departure"], home
        require(now >= max(truck["shift_start_hour"], depot["open_hour"]), "early departure")
        onboard, pickups, deliveries, recorded_order = [], [], [], []
        miles = driving = since_break = 0.0
        for action in route["actions"]:
            kind, start, end = action["kind"], action["start"], action["end"]
            require(abs(start - now) < 1e-6 and end >= start, "schedule gap or overlap")
            if kind == "drive":
                require(action["origin"] == location, "disconnected route")
                destination = action["destination"]
                distance, duration = lanes[location, destination]
                require(abs(end - start - duration) < 1e-6, "incorrect driving time")
                miles += distance
                driving += duration
                since_break += duration
                require(since_break <= limits["break_after_drive_hours"] + 1e-6, "missing driver break")
                weighted_weight += distance * sum(shipments_check[i]["weight_lb"] for i in onboard) / equipment["weight_capacity_lb"]
                weighted_feet += distance * sum(shipments_check[i]["linear_feet"] for i in onboard) / equipment["length_capacity_feet"]
                location = destination
            else:
                require(action["location"] == location, "activity at the wrong location")
                if kind == "break":
                    require(end - start >= limits["break_hours"] - 1e-6, "break too short")
                    since_break = 0.0
                elif kind in ("P", "D"):
                    i = action["shipment"]
                    require(i in shipments_check, "unknown shipment")
                    s = shipments_check[i]
                    prefix = "pickup" if kind == "P" else "delivery"
                    target = s["origin_id"] if kind == "P" else s["destination_id"]
                    facility = facilities_check[target]
                    require(day == s["day"] and location == target, f"wrong day/location for {i}")
                    require(start >= max(s[prefix + "_open_hour"], facility["open_hour"]) - 1e-6
                            and end <= min(s[prefix + "_close_hour"], facility["close_hour"]) + 1e-6,
                            f"service window for {i}")
                    require(abs(end - start - s[prefix + "_service_hours"]) < 1e-6, "handling duration")
                    if kind == "P":
                        require(i not in pickups, "duplicate pickup")
                        require(s["temperature"] != "chilled" or equipment["chilled_capable"] == 1, "wrong equipment")
                        pickups.append(i)
                        onboard.append(i)
                        served[i] += 1
                    else:
                        require(i in onboard, "delivery before pickup or duplicate delivery")
                        require(not parameters["lifo_required"] or onboard[-1] == i, "LIFO unloading order")
                        onboard.remove(i)
                        deliveries.append(i)
                    recorded_order.append((i, kind))
                    for field, capacity in [("weight_lb", "weight_capacity_lb"),
                                             ("linear_feet", "length_capacity_feet"), ("pallets", "pallet_capacity")]:
                        require(sum(shipments_check[j][field] for j in onboard) <= equipment[capacity] + 1e-6, field + " capacity")
                    require(len({(shipments_check[j]["temperature"], shipments_check[j]["compatibility_group"])
                                 for j in onboard}) <= 1, "incompatible freight aboard together")
                else:
                    require(kind == "wait", "unknown activity")
            now = end

        require(location == home and not onboard, "truck must return home empty")
        expected_locations = [home] + [shipments_check[i]["origin_id" if kind == "P" else "destination_id"]
                                      for i, kind in recorded_order] + [home]
        actual_legs = [(a["origin"], a["destination"]) for a in route["actions"] if a["kind"] == "drive"]
        require(actual_legs == list(zip(expected_locations, expected_locations[1:])), "extra or missing travel legs")
        require(pickups and Counter(pickups) == Counter(deliveries) == Counter(route["shipments"]), "route shipment list")
        require(recorded_order == [tuple(event) for event in route["sequence"]], "route stop sequence")
        require(len(pickups) + len(deliveries) <= parameters["max_stops"], "too many stops")
        require(now <= min(truck["shift_end_hour"], depot["close_hour"], 24) + 1e-6, "late depot return")
        require(24 * day + now < 24 * parameters["horizon_end_day"], "outside horizon")
        require(driving <= limits["max_drive_hours"] + 1e-6, "daily driving limit")
        duty = now - route["departure"]
        require(duty <= limits["max_duty_hours"] + 1e-6, "duty limit")
        for field, actual in [("miles", miles), ("drive_hours", driving), ("duty_hours", duty), ("return_time", now)]:
            require(abs(route[field] - actual) < 1e-6, "incorrect " + field)
        previous_tours.setdefault(k, []).append((24 * day + route["departure"], 24 * day + now))
        price = prices[truck["equipment_id"]]
        linehaul = max(price["minimum_linehaul_cad"], miles * price["linehaul_cad_per_mile"])
        stop_cost = max(0, len(recorded_order) - 2) * price["extra_stop_cad"]
        for field, actual in [("linehaul", linehaul), ("stop_cost", stop_cost), ("cost", linehaul + stop_cost)]:
            require(abs(route[field] - actual) < 0.01, "incorrect " + field)
        total_miles += miles
        tours.append(dict(tractor=k, day=day, linehaul=linehaul, stop_cost=stop_cost))

    require(served == Counter({i: 1 for i in shipments_check}), "missing, duplicated or unknown shipments")
    for intervals in previous_tours.values():
        intervals.sort()
        for previous, following in zip(intervals, intervals[1:]):
            require(following[0] - previous[1] >= limits["min_rest_hours"] - 1e-6, "between-day rest")
    linehaul = sum(t["linehaul"] for t in tours)
    stop_cost = sum(t["stop_cost"] for t in tours)
    external_cost = sum(external_costs[i] for i in external)
    return dict(linehaul=linehaul, stop_cost=stop_cost, ltl_cost=external_cost,
                total_cost=linehaul + stop_cost + external_cost, baseline=sum(external_costs.values()),
                miles=total_miles, weight_utilization=weighted_weight / total_miles if total_miles else 0,
                feet_utilization=weighted_feet / total_miles if total_miles else 0)

In [9]:
checked = check_plan(selected_routes, ltl_shipments, data)
if abs(checked["total_cost"] - model.ObjVal) > 0.01:
    raise ValueError("Checked cost does not match the Gurobi objective.")
print(f"PASS: all {len(S)} shipments served exactly once; routes, schedules and costs verified.")

PASS: all 60 shipments served exactly once; routes, schedules and costs verified.


## 8. Final plan and savings

Report the independently checked cost and savings, then show the truck plan. Utilization is averaged over miles traveled, including empty depot legs; the solver gap applies only to the generated candidates.

In [10]:
def clock(hour):
    minutes = round(hour * 60)
    return f"{minutes // 60:02d}:{minutes % 60:02d}"

savings = checked["baseline"] - checked["total_cost"]
savings_pct = 100 * savings / checked["baseline"] if checked["baseline"] else 0
consolidated = sum(len(route["shipments"]) for route in selected_routes if len(route["shipments"]) >= 2)
print(f"Dataset: {DATA_FILE.name}")
print(f"All-LTL baseline:       CAD ${checked['baseline']:,.2f}")
print(f"Truck linehaul:         CAD ${checked['linehaul']:,.2f}")
print(f"Extra-stop charges:     CAD ${checked['stop_cost']:,.2f}")
print(f"Remaining LTL:          CAD ${checked['ltl_cost']:,.2f}")
print(f"FINAL TOTAL:            CAD ${checked['total_cost']:,.2f}")
print(f"SAVINGS:                CAD ${savings:,.2f} ({savings_pct:.2f}%)")
print(f"Checker: PASS | Shipments on shared tours: {consolidated}/{len(S)} ({consolidated / len(S):.1%})")
print(f"Truck shipments: {len(S) - len(ltl_shipments)} | LTL shipments: {len(ltl_shipments)}")
print(f"Weight utilization: {checked['weight_utilization']:.1%} | Floor-space utilization: {checked['feet_utilization']:.1%}")
print(f"Generation: {generation_seconds:.1f}s | Solver: {model.Runtime:.1f}s | Total: {perf_counter() - started:.1f}s")
print(f"Candidate-pool solver gap: {model.MIPGap:.2%}")
print("Candidates were limited by insertion, partner/expansion limits and scheduling choices; no global-optimum claim.")

plan_rows, schedule_rows = [], []
for number, route in enumerate(sorted(selected_routes, key=lambda r: (r["day"], r["tractor"])), 1):
    plan_rows.append(dict(route=number, day=route["day"], truck=route["tractor"],
                          driver=K[route["tractor"]]["driver_name"], shipments=", ".join(route["shipments"]),
                          stops=" → ".join(f"{kind}:{i}" for i, kind in route["sequence"]),
                          departure=clock(route["departure"]), return_time=clock(route["return_time"]),
                          cost_cad=round(route["cost"], 2)))
    for action in route["actions"]:
        schedule_rows.append(dict(route=number, day=route["day"], truck=route["tractor"], **action))
plan_table = pd.DataFrame(plan_rows, columns=["route", "day", "truck", "driver", "shipments", "stops", "departure", "return_time", "cost_cad"])
schedule_table = pd.DataFrame(schedule_rows, columns=["route", "day", "truck", "kind", "location", "origin", "destination", "shipment", "start", "end"])
ltl_table = pd.DataFrame([dict(shipment=i, day=S[i]["day"], cost_cad=round(ltl_cost[i], 2)) for i in ltl_shipments],
                         columns=["shipment", "day", "cost_cad"])
display(plan_table)

output = Path("results-vassili") / DATA_FILE.stem
output.mkdir(parents=True, exist_ok=True)
plan_table.to_csv(output / "truck_routes.csv", index=False)
schedule_table.to_csv(output / "schedule.csv", index=False)
ltl_table.to_csv(output / "ltl_shipments.csv", index=False)
saved_plan = {"routes": selected_routes, "ltl_shipments": ltl_shipments}
(output / "plan.json").write_text(json.dumps(saved_plan, indent=2))
# Verify the saved plan too, so serialization cannot hide a mismatch.
reopened = json.loads((output / "plan.json").read_text())
check_plan(reopened["routes"], reopened["ltl_shipments"], data)
(output / "summary.json").write_text(json.dumps(dict(checked, savings=savings, savings_pct=savings_pct,
    candidate_count=len(routes), solver_gap=model.MIPGap, generation_seconds=generation_seconds,
    solver_seconds=model.Runtime, checker="PASS", scope="Generated candidates only"), indent=2))
print(f"Saved and rechecked plan, schedule, LTL list and summary in {output}.")

Dataset: project1_simplified.xlsx
All-LTL baseline:       CAD $93,298.35
Truck linehaul:         CAD $21,600.00
Extra-stop charges:     CAD $2,800.00
Remaining LTL:          CAD $11,846.15
FINAL TOTAL:            CAD $36,246.15
SAVINGS:                CAD $57,052.20 (61.15%)
Checker: PASS | Shipments on shared tours: 52/60 (86.7%)
Truck shipments: 52 | LTL shipments: 8
Weight utilization: 26.5% | Floor-space utilization: 28.2%
Generation: 0.3s | Solver: 0.0s | Total: 0.5s
Candidate-pool solver gap: 0.00%
Candidates were limited by insertion, partner/expansion limits and scheduling choices; no global-optimum claim.


,route,day,truck,driver,shipments,stops,departure,return_time,cost_cad
0,1,1,T01,Alex Martin,"S001, S002",P:S001 → P:S002 → D:S002 → D:S001,07:00,16:00,1000
1,2,1,T02,Jordan Lee,"S003, S010",P:S003 → P:S010 → D:S003 → D:S010,07:00,16:45,1000
2,3,1,T03,Samira Patel,"S014, S017",P:S014 → P:S017 → D:S017 → D:S014,06:00,14:30,1000
3,4,1,T04,Chris Nguyen,"S011, S018, S020",P:S011 → P:S020 → D:S011 → P:S018 → D:S018 → D...,06:30,16:30,1100
4,5,1,T05,Morgan Blake,"S006, S009",P:S006 → P:S009 → D:S009 → D:S006,07:15,16:15,1000
5,6,1,T06,Riley Chen,"S004, S005, S019",P:S019 → P:S005 → P:S004 → D:S004 → D:S019 → D...,06:15,17:45,1100
6,7,1,T07,Avery Dubois,"S012, S015",P:S015 → P:S012 → D:S012 → D:S015,08:15,17:00,1000
7,8,1,T08,Quinn Rossi,"S013, S016",P:S013 → P:S016 → D:S016 → D:S013,06:00,14:45,1000
8,9,2,T01,Alex Martin,"S021, S030",P:S030 → P:S021 → D:S030 → D:S021,06:30,17:15,1000
9,10,2,T02,Jordan Lee,"S033, S040",P:S033 → P:S040 → D:S040 → D:S033,06:00,17:15,1000


Saved and rechecked plan, schedule, LTL list and summary in results-vassili/project1_simplified.
